**Deep Convolutional GAN (DCGAN)**
- Goal: Understand adversarial training and the two-network min-max game.
- Task: Train a DCGAN on the CIFAR-10 dataset.
- Implementation Details:
-    Generator (G): Takes a random noise vector z  and uses transposed convolutions to generate a 32 x 32 x 3 image.
-    Discriminator (D): A standard CNN binary classifier outputting a probability (using Sigmoid) of the image being real.
-    Training Loop: Carefully implement the alternating training phase using nn.BCELoss. Train D on a batch of real images (labels=1) and fake images (labels=0). Then train G by passing fake images to D but calculating loss with labels=1 (trying to fool D).
-    Evaluation: Track the losses. GANs are notoriously unstable; look for signs of mode collapse (where G outputs the exact same image).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as dset
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# --- Hyperparameters ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 0.0002
beta1 = 0.5  # Standard for DCGAN
batch_size = 128
nz = 100     # Size of latent vector z
nc = 3       # Number of channels (RGB)
ndf = 64     # Discriminator feature map size
ngf = 64     # Generator feature map size
epochs = 25

# --- Data Loading ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
dataset = dset.CIFAR10(root='./data', download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# --- Weight Initialization ---
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# --- Generator Network ---
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # Input is z, going into a convolution
            nn.ConvTranspose2d(nz, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # State size: (ngf*4) x 4 x 4
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # State size: (ngf*2) x 8 x 8
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # State size: (ngf) x 16 x 16
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
            # Output size: (nc) x 32 x 32
        )

    def forward(self, input):
        return self.main(input)

# --- Discriminator Network ---
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # Input is (nc) x 32 x 32
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # State size: (ndf) x 16 x 16
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # State size: (ndf*2) x 8 x 8
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # State size: (ndf*4) x 4 x 4
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input).view(-1)

# --- Initialize Models, Loss, and Optimizers ---
netG = Generator().to(device)
netG.apply(weights_init)

netD = Discriminator().to(device)
netD.apply(weights_init)

criterion = nn.BCELoss()

optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

# --- Training Loop ---
print("Starting Training Loop...")
for epoch in range(epochs):
    for i, data in enumerate(dataloader, 0):
        
        # ==========================================
        # 1. Update Discriminator: max log(D(x)) + log(1 - D(G(z)))
        # ==========================================
        netD.zero_grad()
        
        # Train with all-real batch
        real_cpu = data[0].to(device)
        b_size = real_cpu.size(0)
        label = torch.full((b_size,), 1.0, dtype=torch.float, device=device)
        
        output = netD(real_cpu)
        errD_real = criterion(output, label)
        errD_real.backward()
        
        # Train with all-fake batch
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake = netG(noise)
        label.fill_(0.0)
        
        output = netD(fake.detach()) # Detach to avoid graphing gradients through G
        errD_fake = criterion(output, label)
        errD_fake.backward()
        
        errD = errD_real + errD_fake
        optimizerD.step()

        # ==========================================
        # 2. Update Generator: max log(D(G(z)))
        # ==========================================
        netG.zero_grad()
        label.fill_(1.0) # Fake labels are real for generator cost
        
        output = netD(fake)
        errG = criterion(output, label)
        errG.backward()
        
        optimizerG.step()

        # Output training stats
        if i % 100 == 0:
            print(f'[{epoch}/{epochs}][{i}/{len(dataloader)}] '
                  f'Loss_D: {errD.item():.4f} Loss_G: {errG.item():.4f}')

**Wasserstein GAN with Gradient Penalty (WGAN-GP)**
- Goal: Solve the vanishing gradient and mode collapse issues of standard GANs using Wasserstein distance.
- Task: Upgrade your DCGAN to a WGAN-GP.
- Implementation Details:
-    Remove the Sigmoid activation from the Discriminator (now called a "Critic", outputting a raw real-number score).
-    Loss Formulation: Maximize E[D(x)] - E[D(G(z))]. (In PyTorch: loss_D = torch.mean(D_fake) - torch.mean(D_real)).
-    Gradient Penalty: Instead of weight clipping, calculate the norm of the gradients of the Critic's output with respect to random interpolations between real and fake images. Penalize gradients that deviate from 1. 
-    Evaluation: You should notice that the WGAN-GP loss actually correlates with image quality, unlike standard GAN loss.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as dset
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# --- Hyperparameters ---
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
lr = 0.0001
beta1 = 0.0     # WGAN-GP prefers 0.0 or 0.5 for stability
beta2 = 0.9     # WGAN-GP prefers 0.9
batch_size = 64 # Smaller batch size often used due to GP memory overhead
nz = 100
nc = 3
ndf = 64
ngf = 64
epochs = 25
lambda_gp = 10  # Standard weight for gradient penalty

# --- Data Loading ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
dataset = dset.CIFAR10(root='./data', download=False, transform=transform) # Assumes already downloaded
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# --- Generator Network (Unchanged structure) ---
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, input): return self.main(input)

# --- Critic Network (SIGMOID REMOVED) ---
class Critic(nn.Module):
    def __init__(self):
        super(Critic, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            # Note: WGAN-GP explicitly forbids standard BatchNorm in the Critic 
            # because it introduces intra-batch dependencies. LayerNorm or InstanceNorm is preferred.
            nn.InstanceNorm2d(ndf * 2, affine=True), 
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False)
            # No Sigmoid layer here
        )
    def forward(self, input):
        return self.main(input).view(-1)

# --- Gradient Penalty Function ---
def compute_gradient_penalty(critic, real_samples, fake_samples):
    # Random weight term alpha for interpolation between real and fake samples
    alpha = torch.rand((real_samples.size(0), 1, 1, 1), device=device)
    # Get random interpolation between real and fake samples
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    
    critic_interpolates = critic(interpolates)
    fakes = torch.ones(critic_interpolates.shape, device=device, requires_grad=False)
    
    # Get gradients w.r.t. interpolates
    gradients = torch.autograd.grad(
        outputs=critic_interpolates,
        inputs=interpolates,
        grad_outputs=fakes,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

# --- Init ---
netG = Generator().to(device)
netC = Critic().to(device)

# WGAN-GP uses Adam with customized betas
optimizerC = optim.Adam(netC.parameters(), lr=lr, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, beta2))

# --- Training Loop ---
print("Starting WGAN-GP Training Loop...")
for epoch in range(epochs):
    for i, data in enumerate(dataloader, 0):
        real_images = data[0].to(device)
        b_size = real_images.size(0)
        
        # ==========================================
        # 1. Train Critic (Often trained n_critic=5 times per G step)
        # ==========================================
        netC.zero_grad()
        
        # Format noise vectors
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake_images = netG(noise)
        
        # Scores for real and fake images
        critic_real = netC(real_images)
        critic_fake = netC(fake_images.detach())
        
        # Calculate Gradient Penalty
        gp = compute_gradient_penalty(netC, real_images, fake_images.detach())
        
        # Wasserstein Loss formulation
        loss_C = critic_fake.mean() - critic_real.mean() + lambda_gp * gp
        loss_C.backward()
        optimizerC.step()
        
        # ==========================================
        # 2. Train Generator
        # ==========================================
        if i % 5 == 0:  # Train G less frequently than C
            netG.zero_grad()
            
            # Generate new fakes to clear stale graph states
            gen_fake = netC(netG(noise))
            loss_G = -gen_fake.mean()
            
            loss_G.backward()
            optimizerG.step()

        if i % 100 == 0:
            # Wasserstein Distance estimate: lower (more negative) means better convergence
            w_distance = critic_real.mean() - critic_fake.mean()
            print(f'[{epoch}/{epochs}][{i}/{len(dataloader)}] '
                  f'Loss_C: {loss_C.item():.4f} Loss_G: {loss_G.item():.4f} '
                  f'Wasserstein Dist: {w_distance.item():.4f}')

**Image-to-Image Translation (Pix2Pix)**
- Goal: Conditional GANs mapping one image domain to another.
- Task: Implement Pix2Pix (e.g., mapping satellite images to map routes, or sketches to colored images).
- Implementation Details:
- Generator: Implement a U-Net architecture with skip connections to preserve spatial details from the input image to the output image.
- Discriminator (PatchGAN): Instead of outputting one scalar, the discriminator outputs an N x N matrix where each pixel represents the real/fake prediction of a specific "patch" of the image.
- Loss: Combine the standard Adversarial Loss with an L1 Loss (comparing generated image to the ground truth target image) to enforce low-frequency correctness.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- Hyperparameters ---
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
lr = 0.0002
beta1 = 0.5
beta2 = 0.999
lambda_l1 = 100  # Paper recommends lambda=100 to prioritize structural alignment

# --- U-Net Building Blocks ---
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, submodule=None, innermost=False, outermost=False):
        super(UNetBlock, self).__init__()
        self.outermost = outermost
        
        # Downsample convolutions
        downconv = nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = nn.BatchNorm2d(out_channels)
        
        # Upsample convolutions
        uprelu = nn.ReLU(True)
        upnorm = nn.BatchNorm2d(in_channels)

        if outermost:
            upconv = nn.ConvTranspose2d(out_channels * 2, in_channels, kernel_size=4, stride=2, padding=1)
            down = [downconv]
            up = [uprelu, upconv, nn.Tanh()]
            model = down + [submodule] + up
        elif innermost:
            upconv = nn.ConvTranspose2d(out_channels, in_channels, kernel_size=4, stride=2, padding=1, bias=False)
            down = [downrelu, downconv]
            up = [uprelu, upconv, upnorm]
            model = down + up
        else:
            upconv = nn.ConvTranspose2d(out_channels * 2, in_channels, kernel_size=4, stride=2, padding=1, bias=False)
            down = [downrelu, downconv, downnorm]
            up = [uprelu, upconv, upnorm]
            model = down + [submodule] + up

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        else:
            # Core Skip Connection step: concatenate encoder features with decoder features along channels
            return torch.cat([x, self.model(x)], 1)

# --- Generator (U-Net) ---
class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, num_filters=64):
        super(UNetGenerator, self).__init__()
        # Build U-Net from the inside out
        unet_block = UNetBlock(num_filters * 8, num_filters * 8, submodule=None, innermost=True)
        
        # Add intermediate blocks (bottleneck layers)
        for _ in range(3):
            unet_block = UNetBlock(num_filters * 8, num_filters * 8, submodule=unet_block)
            
        # Gradually upscale channels back down to match input
        unet_block = UNetBlock(num_filters * 4, num_filters * 8, submodule=unet_block)
        unet_block = UNetBlock(num_filters * 2, num_filters * 4, submodule=unet_block)
        unet_block = UNetBlock(num_filters, num_filters * 2, submodule=unet_block)
        
        # Outermost wrapper
        self.model = UNetBlock(in_channels, out_channels, submodule=unet_block, outermost=True)

    def forward(self, input):
        return self.model(input)

# --- PatchGAN Discriminator ---
class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=6, ndf=64): # takes 6 channels: concat(input_img, target_img)
        super(PatchGANDiscriminator, self).__init__()
        self.model = nn.Sequential(
            # Input: (3 + 3) x 256 x 256
            nn.Conv2d(in_channels, ndf, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, True),
            
            # State: ndf x 128 x 128
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, True),
            
            # State: (ndf*2) x 64 x 64
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, True),
            
            # State: (ndf*4) x 32 x 32
            nn.Conv2d(ndf * 4, ndf * 8, kernel_size=4, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, True),
            
            # Final output mapping matrix (Patch mapping layer)
            # State: (ndf*8) x 31 x 31
            nn.Conv2d(ndf * 8, 1, kernel_size=4, stride=1, padding=1),
            nn.Sigmoid()
            # Output shape: Batch_Size x 1 x 30 x 30 (Each unit represents a 70x70 receptive patch field)
        )

    def forward(self, input_img, target_img):
        # PatchGAN expects conditional inputs stacked together along the channel dimension
        x = torch.cat([input_img, target_img], dim=1)
        return self.model(x)

# --- Initialization ---
netG = UNetGenerator().to(device)
netD = PatchGANDiscriminator().to(device)

criterionGAN = nn.BCELoss()
criterionL1 = nn.L1Loss()

optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, beta2))

# --- Single Training Step Breakdown ---
def train_step(real_src, real_tgt):
    batch_size = real_src.size(0)
    
    # ----------------------------------------
    # 1. Update Discriminator (PatchGAN)
    # ----------------------------------------
    optimizerD.zero_grad()
    
    # Train with Real pairs
    pred_real = netD(real_src, real_tgt)
    # Target shape matches PatchGAN output dimensions matrix (filled with 1s)
    label_real = torch.ones_like(pred_real, device=device)
    loss_D_real = criterionGAN(pred_real, label_real)
    
    # Train with Fake pairs
    fake_tgt = netG(real_src)
    pred_fake = netD(real_src, fake_tgt.detach())
    label_fake = torch.zeros_like(pred_fake, device=device)
    loss_D_fake = criterionGAN(pred_fake, label_fake)
    
    # Combine losses
    loss_D = (loss_D_real + loss_D_fake) * 0.5
    loss_D.backward()
    optimizerD.step()
    
    # ----------------------------------------
    # 2. Update Generator (U-Net)
    # ----------------------------------------
    optimizerG.zero_grad()
    
    # Adversarial target (fool discriminator to accept fake targets conditional on sources)
    pred_fake_G = netD(real_src, fake_tgt)
    loss_G_GAN = criterionGAN(pred_fake_G, label_real)
    
    # L1 Loss element to prevent abstract artifact configurations
    loss_G_L1 = criterionL1(fake_tgt, real_tgt) * lambda_l1
    
    loss_G = loss_G_GAN + loss_G_L1
    loss_G.backward()
    optimizerG.step()
    
    return loss_D.item(), loss_G_GAN.item(), loss_G_L1.item()